<a href="https://colab.research.google.com/github/mehmoodanas/Bootcamp/blob/main/Day-9/Pinecone_vector_database_practical_tasks_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [2]:
# place your code here
!pip install pinecone sentence-transformers

From 'pinecone' library import Pinecone, ServerlessSpec.

In [3]:
# place your code here
from pinecone import Pinecone, ServerlessSpec

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [6]:
from google.colab import userdata
from sentence_transformers import SentenceTransformer
import time

# 1. Initialize a client (key from Colab Secrets)
pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))

# 2. Create an index
index_name = "bootcamp"
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='cosine',
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)   # capital I

# 3. Upsert data
texts = [
    "Today we studied RAG, vector search and vector databases.",
    "On the first day of the bootcamp we learned data cleaning.",
    "Our mentors are very helpful and answer all our questions.",
    "Riga is the capital of Latvia with a beautiful old town.",
    "Regular exercise improves heart health and helps you sleep better.",
]
embeddings = model.encode(texts)

vectors = []
for i, (text, emb) in enumerate(zip(texts, embeddings)):
    vectors.append({"id": f"text-{i}", "values": emb.tolist(), "metadata": {"text": text}})

index.upsert(vectors=vectors, namespace="bootcamp")
time.sleep(10)   # wait until vectors are searchable

# 4. Perform a search
my_query = "What did we learn about databases for embeddings?"
query_vector = model.encode(my_query).tolist()

results = index.query(
    namespace="bootcamp",      # same namespace as the upsert
    vector=query_vector,
    top_k=3,
    include_metadata=True,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Print search results.

In [7]:
# place your code here
print("Query:", my_query, "\n")
for match in results["matches"]:
    print(f"{match['score']:.3f}  {match['metadata']['text']}")

Query: What did we learn about databases for embeddings? 

0.452  Today we studied RAG, vector search and vector databases.
0.239  On the first day of the bootcamp we learned data cleaning.
0.185  Our mentors are very helpful and answer all our questions.
